# Exportar YOLOv8n a LiteRT/TFLite para ABCOnlyOne

Este notebook se ejecuta en Google Colab. Genera el archivo `yolov8n_person_fp16.tflite` que luego debe copiarse a Android Studio en:

`app/src/main/assets/yolov8n_person_fp16.tflite`

La app Android no usa internet ni backend. Python/Colab solo se usa una vez para descargar/exportar el modelo.

In [ ]:
!pip install -q ultralytics ai-edge-litert litert-torch

Exportamos YOLOv8n con entrada de 320x320. El modelo original sigue teniendo las 80 clases COCO; en Android se filtra solo la clase `person`, que es el indice 0.

In [ ]:
!yolo export model=yolov8n.pt format=litert imgsz=320

Buscamos el `.tflite` generado y lo renombramos con el nombre que espera la app Android.

In [ ]:
from pathlib import Path
import shutil

candidates = sorted(Path('.').rglob('*.tflite'))
print('Modelos encontrados:')
for path in candidates:
    print(path)

if not candidates:
    raise FileNotFoundError('No se encontro ningun archivo .tflite. Revisa la salida del comando yolo export.')

output = Path('yolov8n_person_fp16.tflite')
shutil.copy(candidates[0], output)
print(f'Archivo listo: {output.resolve()}')

Verificamos las formas de entrada/salida. El lab espera normalmente entrada NCHW `[1, 3, 320, 320]` y salida tipo `[1, 84, 2100]`.

In [ ]:
from ai_edge_litert.interpreter import Interpreter

interpreter = Interpreter(model_path='yolov8n_person_fp16.tflite')
interpreter.allocate_tensors()
print('Input details:')
print(interpreter.get_input_details())
print('\nOutput details:')
print(interpreter.get_output_details())

Descarga el archivo generado. Despues copialo en Android Studio dentro de `app/src/main/assets/`.

In [ ]:
from google.colab import files
files.download('yolov8n_person_fp16.tflite')